# NB15：企業案例 — 企業知識管理系統設計

## 學習目標
- 設計並實作大規模企業知識管理系統
- 掌握文件攝取管道（Ingestion Pipeline）的完整流程
- 實作文件版本管理與知識圖譜構建
- 理解金融業合規需求下的 RAG 系統設計
- 學習跨文件推理與審計日誌的實作方法

## 情境說明
一家大型台灣金融機構（銀行或保險公司）需要建立企業知識管理系統，
處理法規遵循文件、產品手冊、內部政策、研究報告，總計超過 **50 萬頁**。
系統需支援：搜尋、問答、文件摘要、跨文件分析、知識圖譜。

---

## Part 0：系統需求分析

### 0.1 規模需求

| 指標 | 需求 |
|------|------|
| 文件總量 | 50 萬頁 |
| 每日活躍使用者 | 2,000 人 |
| 回應時間 | 3 秒內 |
| 向量索引大小 | 約 1,000 萬個 chunks |
| 並發查詢數 | 200 QPS |

### 0.2 合規需求

- **來源引用**：每個答案必須標注文件來源與版本
- **完整稽核日誌**：記錄每次查詢的使用者、時間戳、檢索文件、答案雜湊
- **資料在地化**：敏感資料不得離境，需部署於本地或私有雲
- **文件時效性**：過期文件需自動標記，不得作為主要答案來源
- **免責聲明注入**：涉及法規主題時自動加入免責聲明

### 0.3 系統架構圖

```
┌─────────────────────────────────────────────────────────────────┐
│                     企業知識管理系統架構                          │
└─────────────────────────────────────────────────────────────────┘

  原始文件
  (PDF/Word/Excel)
       │
       ▼
┌──────────────────┐
│  Ingestion        │  解析 → 清洗 → 語言偵測 → 分塊
│  Pipeline         │  → 嵌入向量 → 實體抽取
└──────┬───────────┘
       │
       ├──────────────────────────┐
       ▼                          ▼
┌──────────────┐         ┌────────────────┐
│  Vector Store │         │ Knowledge Graph │
│  (ChromaDB)   │         │ (Adjacency List)│
│  語意相似搜尋  │         │ 實體關係推理    │
└──────┬───────┘         └───────┬────────┘
       │                          │
       └──────────┬───────────────┘
                  ▼
         ┌────────────────┐
         │  Query Engine   │
         │  多跳檢索        │
         │  Map-Reduce合成 │
         └───────┬────────┘
                 │
                 ▼
         ┌────────────────┐
         │ Compliance Layer│
         │ 引用格式化       │
         │ 稽核日誌        │
         │ 信心分數        │
         │ 免責聲明注入    │
         └───────┬────────┘
                 │
                 ▼
         ┌────────────────┐
         │     使用者      │
         │ (Web/API/Chat) │
         └────────────────┘
```

## Part 1：大規模文件攝取管道（Ingestion Pipeline）

In [ ]:
# 安裝必要套件
# !pip install openai chromadb python-dotenv

In [ ]:
import os
import json
import hashlib
import datetime
import re
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass, field, asdict
from collections import defaultdict

from dotenv import load_dotenv
import openai
import chromadb

load_dotenv()
client = openai.OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

EMBED_MODEL = 'text-embedding-3-small'
LLM_MODEL = 'gpt-4o-mini'

print('✓ 環境初始化完成')
print(f'  嵌入模型：{EMBED_MODEL}')
print(f'  語言模型：{LLM_MODEL}')

In [ ]:
# ── 模擬金融機構文件資料 ──
MOCK_DOCUMENTS = [
    {
        'doc_id': 'REG-2024-001',
        'title': '銀行法第三十六條修正說明',
        'doc_type': 'regulation',
        'source': '金融監督管理委員會',
        'version': '2.1',
        'effective_date': '2024-01-01',
        'expiry_date': '2026-12-31',
        'content': '依據銀行法第三十六條規定，商業銀行辦理放款，其總額不得超過存款總額及金融債券發售額之合計數。'
                   '本次修正重點：一、擴大中小企業放款比率上限至百分之四十。'
                   '二、新增綠色金融放款優先核貸條款。三、強化不動產抵押品評估標準。'
                   '各金融機構應於本規定生效日起九十日內完成內部作業程序修訂，並向主管機關申報。'
    },
    {
        'doc_id': 'PROD-2024-LOAN-001',
        'title': '企業貸款產品手冊 v3.2',
        'doc_type': 'product_manual',
        'source': '授信業務部',
        'version': '3.2',
        'effective_date': '2024-03-01',
        'expiry_date': '2025-02-28',
        'content': '企業貸款產品概述：本行提供多元化企業融資方案，包含短期週轉金貸款、中長期資本支出貸款及聯合授信。'
                   '申請資格：設立滿兩年、無重大違約紀錄之國內登記公司。'
                   '貸款額度：最高新台幣五億元，依企業規模及信用評等核定。'
                   '利率：以台灣銀行基準利率加碼計算，目前加碼範圍為0.5%至3.0%。'
                   '所需文件：最近三年度財務報表、公司登記證明、負責人身分證明。'
    },
    {
        'doc_id': 'POLICY-2024-AML-001',
        'title': '洗錢防制政策暨程序',
        'doc_type': 'internal_policy',
        'source': '法令遵循部',
        'version': '5.0',
        'effective_date': '2024-02-01',
        'expiry_date': '2025-01-31',
        'content': '本行依洗錢防制法及資恐防制法之規定，訂定本政策。'
                   '一、客戶盡職調查（CDD）：所有新開戶客戶均需進行身分驗證及風險評估。'
                   '二、強化盡職調查（EDD）：高風險客戶（包含政治敏感人士PEP）需額外審查。'
                   '三、可疑交易申報（STR）：交易金額超過新台幣五十萬元或具異常特徵者，應於知悉後十個工作日內向調查局申報。'
                   '四、員工訓練：所有員工每年至少完成八小時洗錢防制課程。'
    },
    {
        'doc_id': 'RESEARCH-2024-Q1-001',
        'title': '2024年第一季台灣房市分析報告',
        'doc_type': 'research_report',
        'source': '研究暨策略部',
        'version': '1.0',
        'effective_date': '2024-04-15',
        'expiry_date': '2024-07-14',
        'content': '2024年第一季台灣房市呈現量縮價穩態勢。'
                   '六都交易量較去年同期減少15.3%，主要受高利率環境及選後觀望情緒影響。'
                   '大台北地區新建案推案量達新台幣三千二百億元，年增8%。'
                   '預售屋管制政策持續發酵，換約轉售比率降至歷史低點3.2%。'
                   '展望第二季：預期利率維持高檔，房市交易量難有顯著回升，建議放款審慎評估不動產擔保品現值。'
    },
    {
        'doc_id': 'REG-2024-002',
        'title': '個人資料保護法金融業適用指引',
        'doc_type': 'regulation',
        'source': '金融監督管理委員會',
        'version': '1.3',
        'effective_date': '2024-01-15',
        'expiry_date': '2026-01-14',
        'content': '金融機構蒐集、處理或利用個人資料，應符合個人資料保護法之規定。'
                   '一、蒐集目的：應明確告知當事人蒐集目的，並取得書面同意。'
                   '二、資料最小化：僅蒐集業務必要之最少資料。'
                   '三、資料安全：應建立完善之資訊安全管理制度，防止資料外洩。'
                   '四、當事人權利：應建立機制供當事人行使查詢、更正、刪除等權利。'
                   '五、跨境傳輸：資料傳輸至境外前，應確認該國具備相當個資保護水準。'
    },
    {
        'doc_id': 'PROD-2024-INVEST-001',
        'title': '結構型商品銷售規範',
        'doc_type': 'product_manual',
        'source': '財富管理部',
        'version': '2.0',
        'effective_date': '2024-05-01',
        'expiry_date': '2025-04-30',
        'content': '結構型商品係連結衍生性金融商品之投資工具，具有本金保護或收益增強之特性。'
                   '銷售對象：需通過風險屬性評估，且評估結果為積極型或以上之客戶。'
                   '銷售程序：一、告知產品風險；二、確認客戶適合度；三、說明費用結構；四、取得書面同意。'
                   '冷靜期：客戶得於簽約後三個工作日內無條件解除契約。'
                   '禁止行為：不得對客戶保證獲利或誇大收益預期。'
    },
    {
        'doc_id': 'POLICY-2024-IT-001',
        'title': '資訊安全管理政策',
        'doc_type': 'internal_policy',
        'source': '資訊科技部',
        'version': '4.1',
        'effective_date': '2024-03-15',
        'expiry_date': '2025-03-14',
        'content': '本行資訊安全管理依ISO 27001標準建立，確保資訊的機密性、完整性與可用性。'
                   '一、存取控制：依最小權限原則，員工僅得存取工作所需之系統與資料。'
                   '二、密碼政策：系統密碼長度至少十二位，含大小寫英文、數字及特殊符號，每九十天更換。'
                   '三、端點安全：所有公務電腦均需安裝防毒軟體並定期更新。'
                   '四、事件回應：資安事件應於發現後一小時內通報資安長，嚴重事件需於二十四小時內向主管機關報告。'
    },
    {
        'doc_id': 'RESEARCH-2024-Q2-001',
        'title': '2024年第二季利率展望報告',
        'doc_type': 'research_report',
        'source': '研究暨策略部',
        'version': '1.0',
        'effective_date': '2024-07-01',
        'expiry_date': '2024-09-30',
        'content': '美國聯準會2024年第二季維持利率不變，市場預期年底前降息一至兩碼。'
                   '台灣央行跟進國際趨勢，維持重貼現率1.875%不動。'
                   '台灣十年期公債殖利率於1.5%至1.7%區間波動。'
                   '對本行業務影響：房貸申請量維持低檔，企業融資需求穩定，建議適度增加固定利率商品佔比以鎖定利差。'
    },
    {
        'doc_id': 'REG-2023-001-OLD',
        'title': '銀行法第三十六條原版說明（已廢止）',
        'doc_type': 'regulation',
        'source': '金融監督管理委員會',
        'version': '2.0',
        'effective_date': '2022-01-01',
        'expiry_date': '2023-12-31',
        'content': '依據銀行法第三十六條規定（舊版），商業銀行辦理放款，中小企業放款比率上限為百分之三十五。'
                   '本版本已由REG-2024-001取代，請以最新版本為準。'
    },
    {
        'doc_id': 'POLICY-2024-HR-001',
        'title': '員工行為準則暨道德規範',
        'doc_type': 'internal_policy',
        'source': '人力資源部',
        'version': '3.0',
        'effective_date': '2024-01-01',
        'expiry_date': '2026-12-31',
        'content': '本行員工應遵守誠信、廉潔、專業之行為準則。'
                   '一、利益衝突迴避：員工不得從事與本行業務競爭之活動，持有客戶公司股票超過一定比率需申報。'
                   '二、禁止內線交易：嚴禁利用職務取得之未公開重大資訊從事有價證券交易。'
                   '三、禮品規範：不得收受客戶或供應商超過新台幣五百元之禮品。'
                   '四、申訴機制：員工得透過匿名檢舉系統反映不當行為，本行保證申訴人身分保密。'
    },
    {
        'doc_id': 'PROD-2024-DEPOSIT-001',
        'title': '定期存款產品說明書',
        'doc_type': 'product_manual',
        'source': '存款業務部',
        'version': '1.5',
        'effective_date': '2024-04-01',
        'expiry_date': '2025-03-31',
        'content': '本行定期存款提供一個月至三年之存款期間選擇。'
                   '利率：一年期定存利率為1.845%，二年期1.895%，三年期1.945%（依中央銀行政策調整）。'
                   '存款保險：每一存款人受存款保險保障之最高金額為新台幣三百萬元。'
                   '提前解約：未到期解約者，按原約定利率七折計息。'
                   '適合對象：追求穩定收益、風險承受度低之保守型投資人。'
    },
    {
        'doc_id': 'REG-2024-003',
        'title': '消費者債務清理條例適用說明',
        'doc_type': 'regulation',
        'source': '法務部',
        'version': '1.0',
        'effective_date': '2024-06-01',
        'expiry_date': '2027-05-31',
        'content': '消費者債務清理條例旨在協助無力清償債務之個人進行更生或清算程序。'
                   '更生程序：債務人提出還款計畫，經法院認可後依計畫清償，最長八年。'
                   '清算程序：債務人財產不足清償時，得聲請清算，剩餘債務視情況免除。'
                   '對金融機構影響：銀行應配合法院程序，停止對進入更生程序之債務人執行催收，並依法院核定方案調整還款條件。'
    },
    {
        'doc_id': 'RESEARCH-2024-FINTECH-001',
        'title': '開放銀行第三階段實施評估報告',
        'doc_type': 'research_report',
        'source': '數位金融部',
        'version': '1.0',
        'effective_date': '2024-03-01',
        'expiry_date': '2024-12-31',
        'content': '金管會推動開放銀行第三階段，開放商業交易資訊API供第三方服務供應商（TSP）介接。'
                   '第三階段新增功能：轉帳功能API、信用卡繳費API、貸款申請API。'
                   '資安要求：TSP需取得金管會核可，並符合OAuth 2.0及OpenID Connect標準。'
                   '本行建議：優先開發與主要電商平台之API介接，預估可增加每月新開戶數15%至20%。'
    },
    {
        'doc_id': 'POLICY-2024-CREDIT-001',
        'title': '授信風險管理準則',
        'doc_type': 'internal_policy',
        'source': '風險管理部',
        'version': '6.2',
        'effective_date': '2024-02-15',
        'expiry_date': '2025-02-14',
        'content': '本行授信風險管理依巴塞爾協議III標準建立。'
                   '一、信用評等：採內部評等法（IRB），依違約機率（PD）、違約損失率（LGD）計算風險加權資產。'
                   '二、集中度管理：單一借款人授信上限為本行資本淨額5%，同一集團上限10%。'
                   '三、不動產放款總量管制：依金管會規定，不動產放款餘額不得超過存款總額及金融債券之30%。'
                   '四、逾放比管理：目標維持逾放比低於0.5%，超過0.8%需啟動特別管理機制。'
    },
    {
        'doc_id': 'PROD-2024-FUND-001',
        'title': '境外基金銷售作業準則',
        'doc_type': 'product_manual',
        'source': '財富管理部',
        'version': '2.3',
        'effective_date': '2024-01-01',
        'expiry_date': '2024-12-31',
        'content': '本行代理銷售之境外基金均已取得金管會核准，投資人應詳閱公開說明書。'
                   '適合度評估：依客戶投資屬性（保守、穩健、積極）推介適合商品，不得銷售超出客戶風險承受度之商品。'
                   '資訊揭露：銷售人員應充分說明費用、風險及流動性。'
                   '禁止不當銷售行為：不得以存款或保本名義銷售基金，不得誇大績效。'
                   '申訴處理：客戶如有疑義，得向本行申訴或逕向金融消費評議中心申請評議。'
    },
    {
        'doc_id': 'REG-2024-004',
        'title': '電子支付機構管理條例重點摘要',
        'doc_type': 'regulation',
        'source': '金融監督管理委員會',
        'version': '1.1',
        'effective_date': '2024-07-01',
        'expiry_date': '2027-06-30',
        'content': '電子支付機構管理條例整合電子票券與電子支付平台，擴大業務範圍。'
                   '新增業務：境外小額匯兌、代收代付、儲值帳戶利息給付。'
                   '資本要求：最低實收資本額新台幣三億元，視業務規模可能提高。'
                   '對銀行影響：銀行子公司申請電支執照可共用客戶資料，有助推動數位金融生態系建構。'
    },
    {
        'doc_id': 'POLICY-2024-ESG-001',
        'title': '永續金融暨ESG投資政策',
        'doc_type': 'internal_policy',
        'source': '永續發展辦公室',
        'version': '1.0',
        'effective_date': '2024-01-01',
        'expiry_date': '2026-12-31',
        'content': '本行承諾依赤道原則及聯合國責任銀行原則推動永續金融。'
                   '一、綠色放款：2025年前，綠色金融放款佔總放款比率達10%。'
                   '二、ESG篩選：禁止對煤炭開採、核武製造等敏感產業提供融資。'
                   '三、氣候風險：依TCFD框架揭露氣候相關財務資訊，評估轉型風險與實體風險。'
                   '四、社會責任：每年提撥稅前盈餘1%用於社區發展及金融教育推廣。'
    },
    {
        'doc_id': 'RESEARCH-2024-CREDIT-001',
        'title': '中小企業信用風險評估方法論更新',
        'doc_type': 'research_report',
        'source': '風險管理部',
        'version': '1.0',
        'effective_date': '2024-05-01',
        'expiry_date': '2025-04-30',
        'content': '本報告更新中小企業信用評估模型，納入替代性資料來源以提升預測精準度。'
                   '新增資料維度：電商交易記錄、水電繳費紀錄、社群媒體評分。'
                   '模型績效：新模型Gini係數達0.72，較舊模型提升8個百分點。'
                   '違約率預測：2024年中小企業預期違約率為1.8%，較2023年微升0.2個百分點。'
                   '建議：對替代資料評分低於閾值之客戶，應加強實地訪查，不得單純依賴模型評分核貸。'
    },
    {
        'doc_id': 'PROD-2024-INSURANCE-001',
        'title': '壽險商品銷售資訊揭露準則',
        'doc_type': 'product_manual',
        'source': '保險業務部',
        'version': '1.2',
        'effective_date': '2024-06-01',
        'expiry_date': '2025-05-31',
        'content': '壽險商品銷售應充分揭露保單重要資訊，確保消費者知情決策。'
                   '必要揭露事項：保費結構、解約金損失、保障內容、費用率、業務員佣金揭露。'
                   '招攬規範：不得以存款或投資名義招攬保險，不得對被保險人未盡說明義務。'
                   '保單審閱期：客戶享有至少三日之保單審閱期，審閱期內可無條件退保。'
                   '理賠效率：壽險理賠應於收到完整資料後十五個工作日內完成給付。'
    },
    {
        'doc_id': 'POLICY-2024-BCPLAN-001',
        'title': '營運持續計畫（BCP）',
        'doc_type': 'internal_policy',
        'source': '營運管理部',
        'version': '2.5',
        'effective_date': '2024-04-01',
        'expiry_date': '2025-03-31',
        'content': '本行營運持續計畫涵蓋天然災害、網路攻擊、疫情、系統故障等重大突發事件之因應措施。'
                   '復原時間目標（RTO）：核心銀行系統四小時、ATM網路六小時、網路銀行八小時。'
                   '復原點目標（RPO）：交易資料零遺失，非交易資料最多遺失一小時資料。'
                   '備援機制：主機房位於台北，備援機房位於台中，採即時同步複製。'
                   '演練要求：每年至少執行一次全規模演練，每半年執行一次桌面演練。'
    }
]

print(f'模擬文件數量：{len(MOCK_DOCUMENTS)} 份')
print('文件類型分布：')
from collections import Counter
type_count = Counter(d['doc_type'] for d in MOCK_DOCUMENTS)
for doc_type, count in type_count.items():
    print(f'  {doc_type}: {count} 份')

In [ ]:
@dataclass
class DocumentChunk:
    chunk_id: str
    doc_id: str
    content: str
    chunk_type: str  # 'parent' or 'child'
    parent_chunk_id: Optional[str]
    metadata: Dict


class DocumentIngestionPipeline:
    """大規模文件攝取管道，支援 Parent-Child 分塊策略"""

    def __init__(self, chroma_client: chromadb.Client):
        self.chroma_client = chroma_client
        self.collection = chroma_client.get_or_create_collection(
            name='enterprise_knowledge_base',
            metadata={'hnsw:space': 'cosine'}
        )
        self.processed_docs = []

    # ── Step 1: 解析 ──
    def parse(self, doc: Dict) -> Dict:
        """解析文件（實際場景會處理 PDF/Word/Excel）"""
        return {
            **doc,
            'parsed_content': doc['content'].strip(),
            'char_count': len(doc['content'])
        }

    # ── Step 2: 清洗 ──
    def clean(self, doc: Dict) -> Dict:
        """清洗文字：移除多餘空白、統一標點"""
        text = doc['parsed_content']
        text = re.sub(r'\s+', ' ', text).strip()
        text = text.replace('　', ' ')  # 全形空格
        return {**doc, 'cleaned_content': text}

    # ── Step 3: 語言與類型偵測 ──
    def detect_language_and_type(self, doc: Dict) -> Dict:
        """偵測語言（此處簡化，實際可用 langdetect）"""
        content = doc['cleaned_content']
        # 簡單判斷：中文字符佔比
        chinese_chars = len(re.findall(r'[\u4e00-\u9fff]', content))
        lang = 'zh-TW' if chinese_chars / len(content) > 0.3 else 'en'
        return {**doc, 'language': lang, 'detected_type': doc['doc_type']}

    # ── Step 4: Parent-Child 分塊 ──
    def chunk_parent_child(self, doc: Dict) -> List[DocumentChunk]:
        """Parent-Child 分塊策略：大塊保留語境，小塊精確檢索"""
        content = doc['cleaned_content']
        doc_id = doc['doc_id']
        chunks = []

        # Parent chunk：整份文件作為父塊（實際場景按段落切）
        parent_id = f"{doc_id}__parent__0"
        parent_chunk = DocumentChunk(
            chunk_id=parent_id,
            doc_id=doc_id,
            content=content,
            chunk_type='parent',
            parent_chunk_id=None,
            metadata={
                'doc_id': doc_id,
                'title': doc['title'],
                'doc_type': doc['doc_type'],
                'source': doc['source'],
                'version': doc['version'],
                'effective_date': doc['effective_date'],
                'expiry_date': doc['expiry_date'],
                'chunk_type': 'parent',
                'language': doc.get('language', 'zh-TW')
            }
        )
        chunks.append(parent_chunk)

        # Child chunks：按句號分割，每 2 句為一個子塊
        sentences = [s.strip() for s in re.split(r'(?<=[。！？])', content) if s.strip()]
        child_size = 2
        for i in range(0, len(sentences), child_size):
            child_text = ''.join(sentences[i:i + child_size])
            if len(child_text) < 10:
                continue
            child_id = f"{doc_id}__child__{i // child_size}"
            child_chunk = DocumentChunk(
                chunk_id=child_id,
                doc_id=doc_id,
                content=child_text,
                chunk_type='child',
                parent_chunk_id=parent_id,
                metadata={
                    **parent_chunk.metadata,
                    'chunk_type': 'child',
                    'parent_chunk_id': parent_id,
                    'child_index': i // child_size
                }
            )
            chunks.append(child_chunk)

        return chunks

    # ── Step 5: 嵌入向量 ──
    def embed(self, texts: List[str]) -> List[List[float]]:
        """批次嵌入，實際場景建議 batch size=100"""
        response = client.embeddings.create(
            model=EMBED_MODEL,
            input=texts
        )
        return [item.embedding for item in response.data]

    # ── Step 6: 寫入 ChromaDB ──
    def upsert_to_chromadb(self, chunks: List[DocumentChunk], embeddings: List[List[float]]):
        """將 child chunks 寫入向量資料庫（父塊用於回傳完整語境）"""
        child_chunks = [c for c in chunks if c.chunk_type == 'child']
        child_embeddings = embeddings[:len(child_chunks)]

        if not child_chunks:
            return

        self.collection.upsert(
            ids=[c.chunk_id for c in child_chunks],
            embeddings=child_embeddings,
            documents=[c.content for c in child_chunks],
            metadatas=[c.metadata for c in child_chunks]
        )

    # ── Step 7: 實體抽取（簡化版） ──
    def extract_entities_simple(self, doc: Dict) -> List[str]:
        """從文件標題和內容抽取關鍵實體（簡化版，不呼叫 LLM）"""
        content = doc['cleaned_content'] + ' ' + doc['title']
        # 簡單規則：抓出「部」「法」「條例」「委員會」等關鍵詞前的名詞片語
        patterns = [
            r'[\u4e00-\u9fff]{2,8}(?:委員會|管理局|部)',
            r'[\u4e00-\u9fff]{2,10}(?:法|條例|準則|規範|政策)',
            r'[\u4e00-\u9fff]{2,6}(?:銀行|保險|基金)',
        ]
        entities = []
        for pattern in patterns:
            entities.extend(re.findall(pattern, content))
        return list(set(entities))

    # ── 主流程 ──
    def process_document(self, doc: Dict) -> Dict:
        """執行完整攝取流程"""
        # 1~3: 解析、清洗、偵測
        doc = self.parse(doc)
        doc = self.clean(doc)
        doc = self.detect_language_and_type(doc)

        # 4: 分塊
        chunks = self.chunk_parent_child(doc)

        # 5: 嵌入（只嵌入 child chunks）
        child_chunks = [c for c in chunks if c.chunk_type == 'child']
        child_texts = [c.content for c in child_chunks]

        embeddings = []
        if child_texts:
            embeddings = self.embed(child_texts)

        # 6: 寫入 ChromaDB
        self.upsert_to_chromadb(chunks, embeddings)

        # 7: 實體抽取
        entities = self.extract_entities_simple(doc)

        result = {
            'doc_id': doc['doc_id'],
            'title': doc['title'],
            'language': doc['language'],
            'total_chunks': len(chunks),
            'child_chunks': len(child_chunks),
            'entities': entities[:5],  # 只顯示前 5 個
            'status': 'SUCCESS'
        }
        self.processed_docs.append(result)
        return result


print('DocumentIngestionPipeline 類別定義完成')

In [ ]:
# 初始化 ChromaDB 與攝取管道
chroma_client = chromadb.Client()  # in-memory
pipeline = DocumentIngestionPipeline(chroma_client)

print('開始攝取 20 份金融文件...\n')
print(f'{"文件ID":<25} {"類型":<18} {"語言":<8} {"子塊數":<8} 狀態')
print('─' * 75)

for doc in MOCK_DOCUMENTS:
    result = pipeline.process_document(doc)
    print(f"{result['doc_id']:<25} {doc['doc_type']:<18} {result['language']:<8} "
          f"{result['child_chunks']:<8} {result['status']}")

print('─' * 75)
total_chunks = pipeline.collection.count()
print(f'\n✓ 攝取完成！ChromaDB 向量總數：{total_chunks}')

## Part 2：文件版本管理（Document Versioning）

In [ ]:
class DocumentVersionManager:
    """文件版本管理器：追蹤版本、標記失效、查詢最新有效版本"""

    def __init__(self):
        # {base_doc_id: [版本記錄列表，按時間排序]}
        self.version_registry: Dict[str, List[Dict]] = defaultdict(list)
        self._index_existing_docs()

    def _get_base_id(self, doc_id: str) -> str:
        """從 doc_id 取得基礎 ID（去除版本後綴）"""
        # 例：REG-2024-001-v2 → REG-2024-001
        return re.sub(r'-v\d+$|-OLD$', '', doc_id)

    def _index_existing_docs(self):
        """將現有文件建立版本索引"""
        for doc in MOCK_DOCUMENTS:
            base_id = self._get_base_id(doc['doc_id'])
            # 使用 title 去除版本號後作為 base_id
            title_base = re.sub(r'（已廢止）|v\d+\.\d+|\s+$', '', doc['title']).strip()
            self.register_version(
                base_id=base_id,
                doc_id=doc['doc_id'],
                version=doc['version'],
                effective_date=doc['effective_date'],
                expiry_date=doc['expiry_date'],
                title=doc['title']
            )

    def register_version(self, base_id: str, doc_id: str, version: str,
                          effective_date: str, expiry_date: str, title: str):
        """登記文件版本"""
        record = {
            'doc_id': doc_id,
            'version': version,
            'effective_date': effective_date,
            'expiry_date': expiry_date,
            'title': title,
            'status': 'active',
            'registered_at': datetime.datetime.now().isoformat()
        }
        self.version_registry[base_id].append(record)

    def supersede(self, base_id: str, old_doc_id: str, new_doc: Dict):
        """以新版本取代舊版本"""
        # 標記舊版本為 superseded
        for record in self.version_registry[base_id]:
            if record['doc_id'] == old_doc_id:
                record['status'] = 'superseded'
                record['superseded_by'] = new_doc['doc_id']
                record['superseded_at'] = datetime.datetime.now().isoformat()
                print(f'  ✗ 舊版本 [{old_doc_id} v{record["version"]}] 已標記為 superseded')

        # 登記新版本
        self.register_version(
            base_id=base_id,
            doc_id=new_doc['doc_id'],
            version=new_doc['version'],
            effective_date=new_doc['effective_date'],
            expiry_date=new_doc['expiry_date'],
            title=new_doc['title']
        )
        print(f'  ✓ 新版本 [{new_doc["doc_id"]} v{new_doc["version"]}] 已登記為 active')

    def get_latest_effective(self, base_id: str, as_of_date: str = None) -> Optional[Dict]:
        """查詢指定日期的最新有效版本"""
        if as_of_date is None:
            as_of_date = datetime.date.today().isoformat()

        versions = self.version_registry.get(base_id, [])
        active_versions = [
            v for v in versions
            if v['status'] == 'active'
            and v['effective_date'] <= as_of_date
            and v['expiry_date'] >= as_of_date
        ]

        if not active_versions:
            return None

        # 回傳最新版本（按版本號排序）
        return sorted(active_versions, key=lambda x: x['version'], reverse=True)[0]

    def get_version_history(self, base_id: str) -> List[Dict]:
        """取得完整版本歷史"""
        return self.version_registry.get(base_id, [])

    def is_expired(self, doc_id: str) -> bool:
        """檢查文件是否已過期"""
        today = datetime.date.today().isoformat()
        for versions in self.version_registry.values():
            for v in versions:
                if v['doc_id'] == doc_id:
                    return v['expiry_date'] < today
        return False


print('DocumentVersionManager 類別定義完成')

In [ ]:
version_manager = DocumentVersionManager()

# ── Demo 1：查詢銀行法相關文件的版本歷史 ──
print('═' * 60)
print('Demo 1：版本歷史查詢')
print('═' * 60)

# 檢視 REG-2024-001 的版本資訊
base_id = 'REG-2024-001'
history = version_manager.get_version_history(base_id)
print(f'\n文件 [{base_id}] 版本歷史：')
for v in history:
    status_icon = '✓' if v['status'] == 'active' else '✗'
    print(f'  {status_icon} v{v["version"]} | {v["effective_date"]} ~ {v["expiry_date"]} | {v["status"]}')

# ── Demo 2：模擬法規更新（版本取代） ──
print('\n' + '═' * 60)
print('Demo 2：法規文件版本更新流程')
print('═' * 60)

new_regulation = {
    'doc_id': 'REG-2024-001-v3',
    'title': '銀行法第三十六條修正說明（第三版）',
    'doc_type': 'regulation',
    'source': '金融監督管理委員會',
    'version': '3.0',
    'effective_date': '2025-01-01',
    'expiry_date': '2027-12-31',
    'content': '銀行法第三十六條第三版修正：中小企業放款比率上限調升至百分之四十五，新增氣候相關放款優惠條款。'
}

print(f'\n正在以新版本取代舊版本...')
version_manager.supersede(
    base_id='REG-2024-001',
    old_doc_id='REG-2024-001',
    new_doc=new_regulation
)

# ── Demo 3：查詢特定日期的最新有效版本 ──
print('\n' + '═' * 60)
print('Demo 3：查詢最新有效版本')
print('═' * 60)

for query_date in ['2024-06-01', '2025-06-01', '2026-01-01']:
    latest = version_manager.get_latest_effective('REG-2024-001', as_of_date=query_date)
    if latest:
        print(f'\n查詢日期 {query_date}：')
        print(f'  最新有效版本：{latest["doc_id"]} (v{latest["version"]})')
        print(f'  有效期間：{latest["effective_date"]} ~ {latest["expiry_date"]}')
    else:
        print(f'\n查詢日期 {query_date}：無有效版本')

# ── Demo 4：過期檢查 ──
print('\n' + '═' * 60)
print('Demo 4：文件過期狀態檢查')
print('═' * 60)
check_docs = ['REG-2023-001-OLD', 'POLICY-2024-AML-001', 'REG-2024-001']
for doc_id in check_docs:
    expired = version_manager.is_expired(doc_id)
    status = '⚠ 已過期' if expired else '✓ 有效'
    print(f'  {doc_id}: {status}')

## Part 3：知識圖譜構建（Knowledge Graph）

In [ ]:
@dataclass
class Entity:
    name: str
    entity_type: str  # person, org, product, regulation, concept
    doc_ids: List[str] = field(default_factory=list)


@dataclass
class Relation:
    source: str
    relation: str  # regulates, requires, references, supersedes, applies_to
    target: str
    doc_id: str


class EntityExtractor:
    """使用 LLM 從文件抽取實體"""

    def extract(self, doc: Dict) -> List[Dict]:
        prompt = f"""從以下金融文件中抽取重要實體，回傳 JSON 陣列。
每個實體格式：{{"name": "實體名稱", "type": "org|regulation|product|concept|person"}}

文件標題：{doc['title']}
文件內容（前300字）：{doc['content'][:300]}

只回傳 JSON 陣列，不要其他說明。最多5個實體。"""

        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{'role': 'user', 'content': prompt}],
            temperature=0,
            max_tokens=300
        )
        text = response.choices[0].message.content.strip()
        # 清理可能的 markdown 標記
        text = re.sub(r'^```(?:json)?\n?', '', text)
        text = re.sub(r'\n?```$', '', text)
        try:
            return json.loads(text)
        except:
            return []


class RelationExtractor:
    """使用 LLM 從文件抽取實體關係"""

    def extract(self, doc: Dict, entities: List[Dict]) -> List[Dict]:
        if len(entities) < 2:
            return []

        entity_names = [e['name'] for e in entities]
        prompt = f"""根據以下金融文件，推斷實體間的關係，回傳 JSON 陣列。
每個關係格式：{{"source": "來源實體", "relation": "關係類型", "target": "目標實體"}}
關係類型限：regulates（監管）, requires（要求）, references（引用）, applies_to（適用於）, part_of（屬於）

已抽取實體：{entity_names}
文件內容（前200字）：{doc['content'][:200]}

只回傳 JSON 陣列，最多3個關係。"""

        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{'role': 'user', 'content': prompt}],
            temperature=0,
            max_tokens=200
        )
        text = response.choices[0].message.content.strip()
        text = re.sub(r'^```(?:json)?\n?', '', text)
        text = re.sub(r'\n?```$', '', text)
        try:
            return json.loads(text)
        except:
            return []


class KnowledgeGraph:
    """簡易記憶體知識圖譜（鄰接表實作）"""

    def __init__(self):
        self.entities: Dict[str, Entity] = {}
        self.relations: List[Relation] = []
        # 鄰接表：entity_name → [(relation, target_entity_name)]
        self.adjacency: Dict[str, List[Tuple[str, str]]] = defaultdict(list)

    def add_entity(self, name: str, entity_type: str, doc_id: str):
        if name not in self.entities:
            self.entities[name] = Entity(name=name, entity_type=entity_type, doc_ids=[doc_id])
        elif doc_id not in self.entities[name].doc_ids:
            self.entities[name].doc_ids.append(doc_id)

    def add_relation(self, source: str, relation: str, target: str, doc_id: str):
        self.relations.append(Relation(source=source, relation=relation, target=target, doc_id=doc_id))
        self.adjacency[source].append((relation, target))
        self.adjacency[target].append((f'←{relation}', source))

    def query(self, entity_name: str, max_hops: int = 2) -> Dict:
        """給定實體，找出相關實體與文件（最多 n 跳）"""
        visited = set()
        result = {'entity': entity_name, 'connections': [], 'related_docs': []}

        queue = [(entity_name, 0)]
        while queue:
            current, hop = queue.pop(0)
            if current in visited or hop > max_hops:
                continue
            visited.add(current)

            if current in self.entities:
                for doc_id in self.entities[current].doc_ids:
                    if doc_id not in result['related_docs']:
                        result['related_docs'].append(doc_id)

            for relation, neighbor in self.adjacency.get(current, []):
                result['connections'].append({
                    'from': current, 'relation': relation, 'to': neighbor, 'hop': hop + 1
                })
                queue.append((neighbor, hop + 1))

        return result

    def stats(self) -> Dict:
        return {
            'total_entities': len(self.entities),
            'total_relations': len(self.relations),
            'entity_types': Counter(e.entity_type for e in self.entities.values())
        }


print('知識圖譜相關類別定義完成')

In [ ]:
# 從前 10 份文件構建知識圖譜
kg = KnowledgeGraph()
entity_extractor = EntityExtractor()
relation_extractor = RelationExtractor()

print('正在構建知識圖譜（使用前 10 份文件）...\n')
sample_docs = MOCK_DOCUMENTS[:10]

for doc in sample_docs:
    print(f'處理：{doc["doc_id"]} - {doc["title"][:30]}...')

    # 抽取實體
    entities = entity_extractor.extract(doc)
    for e in entities:
        kg.add_entity(e['name'], e.get('type', 'concept'), doc['doc_id'])

    # 抽取關係
    relations = relation_extractor.extract(doc, entities)
    for r in relations:
        if r.get('source') and r.get('target'):
            kg.add_relation(r['source'], r['relation'], r['target'], doc['doc_id'])

stats = kg.stats()
print(f'\n知識圖譜統計：')
print(f'  實體總數：{stats["total_entities"]}')
print(f'  關係總數：{stats["total_relations"]}')
print(f'  實體類型分布：')
for etype, count in stats['entity_types'].items():
    print(f'    {etype}: {count}')

In [ ]:
# 知識圖譜查詢 Demo
print('═' * 60)
print('知識圖譜查詢 Demo')
print('═' * 60)

# 查詢與「洗錢防制」相關的所有實體與文件
# 從圖譜中找一個已知實體做示範
if kg.entities:
    # 取前幾個實體查詢
    sample_entities = list(kg.entities.keys())[:3]
    for entity_name in sample_entities:
        result = kg.query(entity_name, max_hops=2)
        print(f'\n查詢實體：「{entity_name}」')
        print(f'  相關文件：{result["related_docs"]}')
        if result['connections']:
            print(f'  關係連結（前3條）：')
            for conn in result['connections'][:3]:
                print(f'    [{conn["from"]}] --{conn["relation"]}--> [{conn["to"]}] (第{conn["hop"]}跳)')
else:
    print('\n（知識圖譜為空，請確認 LLM 實體抽取成功）')

## Part 4：跨文件推理（Cross-document Reasoning）

In [ ]:
class CrossDocumentReasoner:
    """跨文件推理引擎：多跳檢索 + Map-Reduce 合成"""

    def __init__(self, collection: chromadb.Collection, version_manager: DocumentVersionManager):
        self.collection = collection
        self.version_manager = version_manager

    def embed_query(self, query: str) -> List[float]:
        response = client.embeddings.create(model=EMBED_MODEL, input=[query])
        return response.data[0].embedding

    def retrieve(self, query: str, n_results: int = 5, filter_expired: bool = True) -> List[Dict]:
        """語意檢索，可過濾過期文件"""
        query_embedding = self.embed_query(query)
        today = datetime.date.today().isoformat()

        # ChromaDB 查詢
        where_filter = None
        if filter_expired:
            where_filter = {'expiry_date': {'$gte': today}}

        results = self.collection.query(
            query_embeddings=[query_embedding],
            n_results=n_results,
            where=where_filter
        )

        docs = []
        for i in range(len(results['ids'][0])):
            docs.append({
                'chunk_id': results['ids'][0][i],
                'content': results['documents'][0][i],
                'metadata': results['metadatas'][0][i],
                'distance': results['distances'][0][i]
            })
        return docs

    def multi_hop_retrieve(self, query: str, hops: int = 2) -> List[Dict]:
        """多跳檢索：第一跳找最相關文件，第二跳用文件摘要擴展查詢"""
        # 第一跳
        first_hop = self.retrieve(query, n_results=3)
        all_results = {r['chunk_id']: r for r in first_hop}

        if hops >= 2 and first_hop:
            # 第二跳：用第一跳結果的關鍵詞擴展查詢
            combined_context = ' '.join([r['content'][:100] for r in first_hop])
            expanded_query = f"{query} {combined_context[:200]}"
            second_hop = self.retrieve(expanded_query, n_results=3)
            for r in second_hop:
                if r['chunk_id'] not in all_results:
                    all_results[r['chunk_id']] = r

        return list(all_results.values())

    def map_reduce_synthesize(self, query: str, retrieved_docs: List[Dict]) -> Dict:
        """Map-Reduce 合成：每份文件獨立摘要後合併"""
        if not retrieved_docs:
            return {'answer': '未找到相關文件。', 'citations': [], 'confidence': 0.0}

        # Map 階段：每份文件獨立提取相關資訊
        map_results = []
        seen_docs = set()
        for doc in retrieved_docs:
            doc_id = doc['metadata'].get('doc_id', 'unknown')
            version = doc['metadata'].get('version', 'N/A')
            if doc_id in seen_docs:
                continue
            seen_docs.add(doc_id)

            map_prompt = f"""根據以下文件片段，提取與問題相關的關鍵資訊。

問題：{query}
文件：[{doc_id} v{version}]
內容：{doc['content']}

請簡要（1-3句話）說明此文件對問題的相關內容。若無關，回答「無相關內容」。"""

            resp = client.chat.completions.create(
                model=LLM_MODEL,
                messages=[{'role': 'user', 'content': map_prompt}],
                temperature=0,
                max_tokens=150
            )
            summary = resp.choices[0].message.content.strip()
            if summary != '無相關內容':
                map_results.append({
                    'doc_id': doc_id,
                    'version': version,
                    'title': doc['metadata'].get('title', ''),
                    'summary': summary
                })

        if not map_results:
            return {'answer': '檢索到的文件與問題無關。', 'citations': [], 'confidence': 0.1}

        # Reduce 階段：合併所有摘要生成最終答案
        sources_text = '\n'.join([
            f"[來源: {r['doc_id']}, 版本{r['version']}] {r['summary']}"
            for r in map_results
        ])

        reduce_prompt = f"""根據以下多份文件的摘要，綜合回答問題。
回答必須在每個重要陳述後標注 [來源: doc_id, 版本X] 格式的引用。

問題：{query}

文件摘要：
{sources_text}

請提供完整、準確的綜合答案（繁體中文，200字以內）："""

        final_resp = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{'role': 'user', 'content': reduce_prompt}],
            temperature=0,
            max_tokens=400
        )
        answer = final_resp.choices[0].message.content.strip()

        citations = [{'doc_id': r['doc_id'], 'version': r['version'], 'title': r['title']} for r in map_results]
        confidence = min(0.95, 0.5 + len(map_results) * 0.15)

        return {'answer': answer, 'citations': citations, 'confidence': confidence}


print('CrossDocumentReasoner 類別定義完成')

In [ ]:
reasoner = CrossDocumentReasoner(pipeline.collection, version_manager)

# 跨文件問答 Demo
compliance_questions = [
    '企業貸款申請需要符合哪些法規要求？有什麼利率和額度限制？',
    '金融機構在個人資料保護和洗錢防制方面有哪些主要義務？',
]

for q in compliance_questions:
    print('═' * 65)
    print(f'問題：{q}')
    print('─' * 65)

    # 多跳檢索
    retrieved = reasoner.multi_hop_retrieve(q, hops=2)
    print(f'檢索到 {len(retrieved)} 個相關片段')

    # Map-Reduce 合成
    result = reasoner.map_reduce_synthesize(q, retrieved)

    print(f'\n回答：')
    print(result['answer'])
    print(f'\n引用來源：')
    for c in result['citations']:
        print(f'  • [{c["doc_id"]}, 版本{c["version"]}] {c["title"]}')
    print(f'\n信心分數：{result["confidence"]:.2f}')
    print()

## Part 5：合規層（Compliance Layer）

In [ ]:
class CitationFormatter:
    """引用格式化器：確保每個答案都有標準化引用格式"""

    @staticmethod
    def format_inline(doc_id: str, version: str, title: str) -> str:
        return f"[來源: {doc_id}, 版本{version}]"

    @staticmethod
    def format_bibliography(citations: List[Dict]) -> str:
        lines = ['\n\n【參考文件】']
        for i, c in enumerate(citations, 1):
            lines.append(f"{i}. {c['title']} ({c['doc_id']}, v{c['version']})")
        return '\n'.join(lines)

    @staticmethod
    def ensure_citations_present(answer: str, citations: List[Dict]) -> str:
        """確保答案中包含引用標記，否則自動附加"""
        if not citations:
            return answer
        if '[來源:' not in answer:
            citation_str = '、'.join([f"[來源: {c['doc_id']}, 版本{c['version']}]" for c in citations])
            answer = answer + f'\n\n（資料來源：{citation_str}）'
        return answer


class AuditLogger:
    """稽核日誌記錄器：記錄每次查詢的完整軌跡"""

    def __init__(self):
        self.logs: List[Dict] = []

    def log(self, user_id: str, query: str, retrieved_docs: List[str],
             answer: str, confidence: float, session_id: str = None) -> str:
        """記錄查詢，回傳 audit_id"""
        audit_id = hashlib.sha256(
            f"{user_id}{query}{datetime.datetime.now().isoformat()}".encode()
        ).hexdigest()[:16]

        answer_hash = hashlib.sha256(answer.encode()).hexdigest()[:16]

        log_entry = {
            'audit_id': audit_id,
            'timestamp': datetime.datetime.now().isoformat(),
            'user_id': user_id,
            'session_id': session_id or 'N/A',
            'query': query,
            'retrieved_doc_ids': retrieved_docs,
            'answer_hash': answer_hash,
            'confidence': confidence,
            'answer_length': len(answer)
        }
        self.logs.append(log_entry)
        return audit_id

    def get_user_history(self, user_id: str) -> List[Dict]:
        return [l for l in self.logs if l['user_id'] == user_id]

    def export_json(self) -> str:
        return json.dumps(self.logs, ensure_ascii=False, indent=2)


class ComplianceLayer:
    """合規層：整合引用格式化、稽核日誌、信心評分、免責聲明"""

    REGULATORY_KEYWORDS = [
        '洗錢', '法規', '條例', '監管', '主管機關', '合規', '申報',
        '違反', '處罰', '金管會', '罰鍰', '裁處'
    ]

    DISCLAIMER = (
        "\n\n⚠ 免責聲明：本回答僅供參考，不構成法律或投資建議。"
        "實際法規遵循事宜請諮詢專業法務人員，"
        "並以主管機關最新公告為準。"
    )

    def __init__(self):
        self.formatter = CitationFormatter()
        self.audit_logger = AuditLogger()

    def is_regulatory_topic(self, query: str) -> bool:
        return any(kw in query for kw in self.REGULATORY_KEYWORDS)

    def compute_confidence(self, raw_confidence: float, citations: List[Dict],
                            query: str) -> Dict:
        """多因子信心評分"""
        score = raw_confidence
        factors = []

        # 因子1：引用數量
        if len(citations) >= 3:
            score += 0.05
            factors.append('多來源佐證 +0.05')
        elif len(citations) == 0:
            score -= 0.20
            factors.append('無來源引用 -0.20')

        # 因子2：是否涉及法規（法規問題風險較高，降低信心）
        if self.is_regulatory_topic(query):
            score -= 0.05
            factors.append('法規主題（需人工確認）-0.05')

        score = max(0.0, min(1.0, score))
        return {'score': round(score, 3), 'factors': factors}

    def process(self, query: str, raw_result: Dict, user_id: str,
                 session_id: str = None) -> Dict:
        """合規層完整處理流程"""
        answer = raw_result['answer']
        citations = raw_result['citations']

        # 1. 確保引用標記存在
        answer = self.formatter.ensure_citations_present(answer, citations)

        # 2. 附加參考文件書目
        if citations:
            answer += self.formatter.format_bibliography(citations)

        # 3. 法規主題注入免責聲明
        if self.is_regulatory_topic(query):
            answer += self.DISCLAIMER

        # 4. 信心評分
        confidence_result = self.compute_confidence(
            raw_result.get('confidence', 0.5), citations, query
        )

        # 5. 稽核日誌
        retrieved_doc_ids = [c['doc_id'] for c in citations]
        audit_id = self.audit_logger.log(
            user_id=user_id,
            query=query,
            retrieved_docs=retrieved_doc_ids,
            answer=answer,
            confidence=confidence_result['score'],
            session_id=session_id
        )

        return {
            'answer': answer,
            'citations': citations,
            'confidence': confidence_result,
            'audit_id': audit_id,
            'regulatory_topic': self.is_regulatory_topic(query)
        }


print('合規層類別定義完成')

In [ ]:
# 合規層整合 Demo
compliance = ComplianceLayer()

test_queries = [
    ('user-001', 'session-A', '企業申請貸款的額度上限是多少？'),
    ('user-002', 'session-B', '洗錢防制法規要求金融機構如何申報可疑交易？'),
    ('user-001', 'session-A', '定期存款的存款保險保障額度是多少？'),
]

for user_id, session_id, query in test_queries:
    print('═' * 65)
    print(f'使用者：{user_id} | 問題：{query}')
    print('─' * 65)

    # 跨文件推理
    retrieved = reasoner.multi_hop_retrieve(query)
    raw_result = reasoner.map_reduce_synthesize(query, retrieved)

    # 合規層處理
    final = compliance.process(query, raw_result, user_id, session_id)

    print(final['answer'])
    print(f'\n信心分數：{final["confidence"]["score"]} | 稽核ID：{final["audit_id"]}')
    if final['regulatory_topic']:
        print('（已自動注入免責聲明）')
    print()

# 顯示稽核日誌
print('\n' + '═' * 65)
print('稽核日誌（user-001 的查詢記錄）')
print('═' * 65)
for log in compliance.audit_logger.get_user_history('user-001'):
    print(f"  [{log['timestamp'][:19]}] 稽核ID: {log['audit_id']}")
    print(f"    查詢：{log['query']}")
    print(f"    引用文件：{log['retrieved_doc_ids']}")
    print(f"    答案雜湊：{log['answer_hash']}")
    print(f"    信心分數：{log['confidence']}")
    print()

## Part 6：知識更新與失效管理

In [ ]:
class DocumentExpiryManager:
    """文件失效管理：偵測過期、標記陳舊、管理重新索引"""

    def __init__(self, collection: chromadb.Collection):
        self.collection = collection

    def scan_expired_documents(self) -> Dict:
        """掃描所有文件，找出過期、即將過期、陳舊的文件"""
        today = datetime.date.today()
        warning_threshold = today + datetime.timedelta(days=30)

        results = {
            'expired': [],
            'expiring_soon': [],   # 30 天內到期
            'active': [],
            'scan_date': today.isoformat()
        }

        seen_docs = set()
        for doc in MOCK_DOCUMENTS:
            doc_id = doc['doc_id']
            if doc_id in seen_docs:
                continue
            seen_docs.add(doc_id)

            expiry = datetime.date.fromisoformat(doc['expiry_date'])
            record = {
                'doc_id': doc_id,
                'title': doc['title'],
                'expiry_date': doc['expiry_date'],
                'doc_type': doc['doc_type']
            }

            if expiry < today:
                record['days_expired'] = (today - expiry).days
                results['expired'].append(record)
            elif expiry <= warning_threshold:
                record['days_until_expiry'] = (expiry - today).days
                results['expiring_soon'].append(record)
            else:
                results['active'].append(record)

        return results

    def remove_expired_from_index(self, expired_doc_ids: List[str]) -> int:
        """從向量索引中移除過期文件的 chunks"""
        removed_count = 0
        for doc_id in expired_doc_ids:
            # 查詢該文件的所有 chunks
            results = self.collection.get(
                where={'doc_id': doc_id}
            )
            if results['ids']:
                self.collection.delete(ids=results['ids'])
                removed_count += len(results['ids'])
                print(f'  已移除 [{doc_id}] 的 {len(results["ids"])} 個向量')
        return removed_count

    def reindex_strategy(self, doc_ids: List[str]) -> Dict:
        """重新索引策略建議"""
        return {
            'strategy': 'incremental',
            'priority': 'regulation > internal_policy > product_manual > research_report',
            'batch_size': 100,
            'estimated_time_minutes': len(doc_ids) * 2,
            'docs_to_reindex': doc_ids,
            'recommendation': (
                '建議在離峰時段（凌晨2~5時）執行重新索引。'
                '優先處理法規文件，確保合規性答案始終使用最新版本。'
            )
        }

    def check_stale_in_retrieval(self, retrieved_docs: List[Dict]) -> List[Dict]:
        """檢查檢索結果中是否包含陳舊文件，並加入警告標記"""
        today = datetime.date.today().isoformat()
        for doc in retrieved_docs:
            expiry = doc['metadata'].get('expiry_date', '9999-12-31')
            if expiry < today:
                doc['stale_warning'] = f'⚠ 此文件已於 {expiry} 過期，請確認是否有更新版本'
            else:
                doc['stale_warning'] = None
        return retrieved_docs


# ── Demo ──
expiry_manager = DocumentExpiryManager(pipeline.collection)

print('═' * 65)
print('文件時效性掃描報告')
print('═' * 65)

scan_result = expiry_manager.scan_expired_documents()
print(f'掃描日期：{scan_result["scan_date"]}')
print(f'\n已過期文件（{len(scan_result["expired"])} 份）：')
for d in scan_result['expired']:
    print(f'  ✗ [{d["doc_id"]}] {d["title"][:30]} | 逾期 {d["days_expired"]} 天')

print(f'\n即將到期文件（30天內，{len(scan_result["expiring_soon"])} 份）：')
for d in scan_result['expiring_soon']:
    print(f'  ⚠ [{d["doc_id"]}] {d["title"][:30]} | 剩餘 {d["days_until_expiry"]} 天')

print(f'\n有效文件：{len(scan_result["active"])} 份')

# 重新索引策略
print('\n' + '═' * 65)
print('重新索引策略建議')
print('═' * 65)
stale_ids = [d['doc_id'] for d in scan_result['expired']]
strategy = expiry_manager.reindex_strategy(stale_ids)
print(f'策略：{strategy["strategy"]}')
print(f'優先順序：{strategy["priority"]}')
print(f'批次大小：{strategy["batch_size"]} 份/批')
print(f'預估時間：{strategy["estimated_time_minutes"]} 分鐘')
print(f'建議：{strategy["recommendation"]}')

## Part 7：面試 Q&A

---

### Q1：如何設計 RAG 系統以處理 50 萬頁的大規模文件庫？

**A：** 大規模文件庫的 RAG 設計需從三個維度考量：

**存儲層：** 採分散式向量資料庫（如 Milvus、Pinecone、Weaviate），以水平擴展支撐千萬級向量。配合 HNSW 近似最近鄰索引，將單次查詢耗時控制在毫秒級。

**分塊策略：** 使用 Parent-Child Chunking：子塊（~200 字）用於精確語意檢索，父塊（~1000 字）提供完整語境給 LLM 生成。此策略既保留語意精準度，又避免上下文碎片化。

**攝取管道：** 以非同步批次處理（Celery/Kafka）處理大量文件，每份文件依序執行解析→清洗→分塊→嵌入→寫入，支援斷點續傳。嵌入使用 batch API 降低延遲與成本。

**快取策略：** 對高頻查詢的嵌入結果做語意快取（Semantic Cache），相似度超過閾值直接回傳快取答案，可降低 30~50% 的 API 呼叫量。

---

### Q2：金融業 RAG 系統如何確保每個答案都有可追溯的來源引用？

**A：** 來源引用需從架構層面強制執行，而非依賴提示詞：

**強制引用格式：** 在 System Prompt 中明確要求每個陳述後附加 `[來源: doc_id, 版本X]` 格式。在後處理層（Compliance Layer）自動偵測答案是否缺少引用標記，若缺少則自動附加「資料來源」區塊。

**元資料傳遞：** 在向量儲存中為每個 chunk 保存完整元資料（doc_id、版本、來源部門、有效日期），確保檢索時元資料隨文件一起回傳。

**Map-Reduce 架構：** Map 階段讓 LLM 對每份文件分別提取資訊並標注來源，Reduce 階段合併時要求保留所有引用。此架構相比單次 Stuffing 更能確保引用完整性。

---

### Q3：知識圖譜與向量搜尋各有什麼優缺點？實際系統中如何搭配使用？

**A：**

| 維度 | 向量搜尋 | 知識圖譜 |
|------|----------|----------|
| 優點 | 語意相似、自然語言友好、擴展性強 | 精確關係推理、多跳查詢、可解釋 |
| 缺點 | 缺乏結構化推理、黑盒不透明 | 建構成本高、難以處理非結構化文字 |
| 適合 | 語意搜尋、FAQ、文件摘要 | 實體關係查詢、影響分析、知識推導 |

**搭配策略（Hybrid RAG）：**
1. 使用者提問 → 先由知識圖譜識別涉及的實體與關係
2. 用實體名稱擴展查詢，再做向量搜尋
3. 將知識圖譜的關係路徑作為 context 注入 LLM
4. 例：問「新版銀行法對企業貸款有何影響？」→ 圖譜識別「銀行法→監管→商業銀行」，擴展後向量搜尋相關條文

---

### Q4：RAG 系統中的文件版本管理應如何設計？

**A：** 文件版本管理是金融業 RAG 的核心挑戰：

**版本狀態機：** 每份文件維護狀態（active / superseded / expired / draft），查詢時預設只返回 active 狀態的最新版本。

**有效日期過濾：** 向量資料庫的 metadata 記錄 `effective_date` 和 `expiry_date`，在 ChromaDB 的 `where` 過濾器中加入日期條件，確保時點一致性。

**版本取代流程：** 新版本上架時，舊版本標記為 superseded（保留供稽核），舊版本的向量 chunks 從檢索索引中移除，新版本重新嵌入。版本變更事件寫入審計日誌。

**歷史版本查詢：** 稽核或回溯分析時允許指定 `as_of_date` 參數，系統返回當時有效的版本，實現時間點查詢（Point-in-Time Query）。

---

### Q5：企業級 RAG 的稽核日誌應記錄哪些資訊？如何實作？

**A：** 金融業稽核日誌需支援監管機關調閱與內部風控審查：

**必要欄位：**
- `audit_id`：不可篡改的唯一識別碼（SHA-256 雜湊）
- `timestamp`：ISO 8601 精確時間戳
- `user_id` + `session_id`：使用者身份追蹤
- `query`：原始問題全文
- `retrieved_doc_ids`：所有檢索到的文件 ID 與版本
- `answer_hash`：答案的 SHA-256 雜湊（確保日誌不被回溯竄改）
- `confidence_score`：答案信心分數
- `model_version`：使用的 LLM 與嵌入模型版本

**實作要點：**
1. 日誌寫入應為 append-only（不可修改），可使用 WORM 儲存
2. 日誌本身需備份至獨立系統，避免主系統受攻擊時日誌遭刪除
3. 支援按使用者、時間範圍、文件 ID 查詢日誌
4. 定期稽核：自動比對答案雜湊，偵測是否有未記錄的系統變更
5. 資料保存期限：依金融業法規通常需保存 5~7 年

## 系統總結

### 本案例涵蓋的核心技術

```
企業知識管理系統 — 技術棧總覽

┌─────────────────────────────────────────────┐
│  攝取管道    │  Parent-Child Chunking          │
│              │  批次嵌入（text-embedding-3-small│
│              │  多步驟處理：解析→清洗→分塊→向量 │
├─────────────────────────────────────────────┤
│  版本管理    │  狀態機（active/superseded）     │
│              │  時間點查詢（Point-in-Time）      │
│              │  有效日期過濾                    │
├─────────────────────────────────────────────┤
│  知識圖譜    │  LLM 實體抽取                   │
│              │  關係抽取（regulates/requires）  │
│              │  鄰接表圖遍歷（多跳查詢）         │
├─────────────────────────────────────────────┤
│  跨文件推理  │  多跳檢索（2-hop retrieval）     │
│              │  Map-Reduce 合成                │
│              │  強制引用格式                   │
├─────────────────────────────────────────────┤
│  合規層      │  引用格式化（CitationFormatter）  │
│              │  稽核日誌（AuditLogger）          │
│              │  多因子信心評分                  │
│              │  免責聲明自動注入                │
├─────────────────────────────────────────────┤
│  失效管理    │  過期文件掃描                   │
│              │  向量索引清理                   │
│              │  增量重新索引策略               │
└─────────────────────────────────────────────┘
```

### 下一步擴展方向

1. **高可用性**：將 ChromaDB 替換為 Milvus Cluster，支援水平擴展至億級向量
2. **多模態**：擴展攝取管道支援 PDF 掃描檔（OCR）、Excel 表格、PowerPoint 投影片
3. **串流更新**：整合 Kafka 事件流，法規變更時自動觸發重新索引
4. **使用者反饋**：建立 RLHF 機制，根據使用者評分持續優化檢索與生成品質
5. **多語言支援**：擴展至英文、日文法規文件，實作跨語言語意搜尋